# Assignment 1 — Smart decisions with Jev (TypeSafe AI)

**Task:** route customer-support messages for *Bazarly*, a (fictional) online marketplace in Kazakhstan, into
`billing · delivery · technical · security · general · spam` with a `high/normal` priority.

**Workflow:** one Jev call asks 4 typed questions → plain Python combines them with thresholds → route or send to a human.
The same 80 labeled tickets + 10 tricky inputs are then sent to a big LLM that must answer in JSON.

**No API keys are stored in this notebook.** Put them in Colab's 🔑 *Secrets* panel (left sidebar) with the names below,
or type them when asked.

| Secret | Needed for |
|---|---|
| `TYPESAFE_API_KEY` | Jev via TypeSafe's API (`JEV_BACKEND = "typesafe"`) |
| `CLOUDFLARE_ACCOUNT_ID`, `CLOUDFLARE_API_TOKEN` | Jev via Cloudflare Workers AI, model `typesafe/jev` (`JEV_BACKEND = "cloudflare"`) |
| `ANTHROPIC_API_KEY` | Claude as the LLM baseline (`LLM_BACKEND = "anthropic"`) |
| `OPENAI_API_KEY` | a GPT model as the baseline (`LLM_BACKEND = "openai"`) |
| `GROQ_API_KEY` | free gpt-oss-120b on Groq as the baseline (`LLM_BACKEND = "groq"`, ~8 s pause between calls for the free limits) |

**Zero-budget option:** `JEV_BACKEND = "cloudflare"`, `LLM_BACKEND = "cloudflare"`, `LLM_MODEL = None` runs Jev and
Llama 3.3 70B on Cloudflare Workers AI with only the two Cloudflare secrets. The whole run needs a few thousand
neurons, inside the 10,000 free per day (check your Workers AI usage page to confirm Jev is covered).


## 0. Settings

In [ ]:
JEV_BACKEND = "typesafe"          # "typesafe" or "cloudflare"
JEV_MODEL   = None                # None -> "jev-1.13.0" (TypeSafe, pinned) / "typesafe/jev" (Cloudflare)
LLM_BACKEND = "anthropic"         # "anthropic", "openai", "cloudflare" (Llama 3.3 70B, free daily quota)
                                  # or "groq" (gpt-oss-120b, free tier, no card)
LLM_MODEL   = "claude-sonnet-5-5" # the "big LLM" baseline; e.g. "claude-haiku-5-5", a GPT model id,
                                  # or None with "cloudflare"/"groq" to use their defaults
DRY_RUN     = False               # True = fake models, no keys, numbers meaningless (only to test the code)

REPO_URL = "https://github.com/<your-username>/jev-ticket-router.git"   # <- your fork / repo

import os
DRY_RUN = DRY_RUN or os.environ.get("JEVBENCH_DRY_RUN") == "1"


## 1. Get the code and install the SDKs

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.exists("jevbench"):
    if not os.path.exists("jev-ticket-router"):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("jev-ticket-router")
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("working dir:", os.getcwd())


## 2. API keys (read from Colab Secrets or typed in; never saved)

In [ ]:
from getpass import getpass

def secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        v = userdata.get(name)
    except Exception:
        v = None
    v = v or getpass(f"{name}: ")
    os.environ[name] = v
    return v

if not DRY_RUN:
    if JEV_BACKEND == "typesafe":
        secret("TYPESAFE_API_KEY")
    if "cloudflare" in (JEV_BACKEND, LLM_BACKEND):
        secret("CLOUDFLARE_ACCOUNT_ID"); secret("CLOUDFLARE_API_TOKEN")
    if LLM_BACKEND == "anthropic":
        secret("ANTHROPIC_API_KEY")
    elif LLM_BACKEND == "openai":
        secret("OPENAI_API_KEY")
    elif LLM_BACKEND == "groq":
        secret("GROQ_API_KEY")
print("keys set:", [k for k in ("TYPESAFE_API_KEY", "CLOUDFLARE_API_TOKEN", "ANTHROPIC_API_KEY", "OPENAI_API_KEY") if os.environ.get(k)])


## 3. The typed questions

| id | type | asks | used by the policy for |
|---|---|---|---|
| `topic` | **Choice** (6 labels) | which team should handle it | the queue |
| `urgency` | **Score** (0–3 rubric) | how urgently a human must respond | priority = high if score ≥ 1.5 |
| `unauthorized_access` | **Noul** | did someone else get into the account / money / codes | safety override → `security` if ≥ 0.5 |
| `injection_attempt` | **Noul** | does the text try to instruct the classifier or fake a system note | always → human if ≥ 0.5 |


In [ ]:
import json
from jevbench.questions import QUESTIONS
from jevbench import config
print(json.dumps(QUESTIONS, indent=2, ensure_ascii=False)[:2500], "...")
print("\nthresholds:", dict(SECURITY_T=config.SECURITY_T, URGENCY_HIGH_T=config.URGENCY_HIGH_T,
                            INJECTION_T=config.INJECTION_T, TARGET_ACCURACY=config.TARGET_ACCURACY))


## 4. One ticket, end to end

In [ ]:
from jevbench.jev import make_jev
from jevbench.llm import make_llm
from jevbench.policy import decide

jev = make_jev("fake" if DRY_RUN else JEV_BACKEND, JEV_MODEL)
llm = make_llm("fake" if DRY_RUN else LLM_BACKEND, LLM_MODEL)

ticket = "Someone logged into my account from another city and placed an order for an iPhone that I didn't make!"
answers, usage, model = jev.ask(ticket)
print("model:", model, "| usage:", usage)
for name, a in answers.items():
    print(f"  {name:<20}", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in a.items() if k != "legend"})
print("decision:", decide(answers))

raw, usage, model = llm.ask(ticket)
print("\nLLM", model, usage, "->", raw)


## 5. Run both systems on the labeled set and the tricky set
Results are cached in `results/*.jsonl`; re-running only calls the API for missing tickets.

In [ ]:
import pandas as pd
from jevbench.runner import run_system

OUT = "results_dryrun" if DRY_RUN else "results"
tickets = pd.read_csv("data/tickets.csv")
tricky = pd.read_csv("data/tricky.csv")
print(tickets.queue.value_counts().to_dict(), "| languages:", tickets.lang.value_counts().to_dict())

for name, df in (("tickets", tickets), ("tricky", tricky)):
    run_system(jev, df, f"{OUT}/jev_{name}.jsonl", verbose=False)
    run_system(llm, df, f"{OUT}/llm_{name}.jsonl", verbose=False)
jev.close(); llm.close()
print("done")


## 6. Results: Jev vs LLM

In [ ]:
from pathlib import Path
from IPython.display import Markdown, Image, display
from run import build_report
from jevbench.report import to_markdown

rep = build_report(Path(OUT), tickets, tricky, dry_run=DRY_RUN)
display(Markdown(to_markdown(rep["table"], index=True)))
display(Markdown("**Queue accuracy by language**\n\n" + to_markdown(rep["langs"].round(3), index=True)))


## 7. Confidence threshold: how much can be automated at ≥ 95 % accuracy?

In [ ]:
display(Image(f"{OUT}/confidence_threshold.png"))
n = len(tickets)
for name, p in rep["points"].items():
    if p:
        print(f"{name}: threshold {p['threshold']:.2f} -> {p['auto_n']}/{n} auto-decided ({p['coverage']:.0%}) "
              f"at {p['accuracy']:.1%} accuracy (95% CI lower bound {p['acc_wilson_low']:.1%}); "
              f"{n - p['auto_n']} need a human")
    else:
        print(f"{name}: no threshold reaches the target accuracy")


## 8. Break it: 10 tricky inputs

In [ ]:
pd.set_option("display.max_colwidth", 120)
rep["tricky"][["id", "trick", "text", "expected", "Jev decision", "Jev ok", "LLM decision", "LLM ok"]]


In [ ]:
rep["tricky"][["id", "trick", "Jev raw"]]   # Jev's underlying typed answers for each tricky input

## 9. Error analysis
Which labeled tickets did Jev route wrongly, and how confident was it?

In [ ]:
from jevbench import report as R
jev_df = R.to_frame(R.load_results(f"{OUT}/jev_tickets.jsonl"), tickets)
jev_df.loc[~jev_df.queue_ok, ["id", "lang", "text", "queue", "pred_queue", "confidence"]].sort_values("confidence", ascending=False)


Everything is also written to `results/results.md` (tables + plot), `results_table.csv`,
`threshold_curve.csv` and `tricky_cases.csv`. Commit the `results/` folder; it contains no keys.